# ✦ LILY WAN 2.2 — Adaptive Kaggle Studio — v4

Choose **GPU T4 x2** if Kaggle honors it. If Kaggle overrides you with a **P100**, this notebook automatically switches to a safe single-P100 profile instead of aborting. ComfyUI dependencies are isolated from the Kaggle kernel, and RIFE is deferred for reliable startup.


In [ ]:
import json, urllib.request, sys, subprocess
from pathlib import Path

print("✦ Lily Wan 2.2 Studio — adaptive startup v4")

try:
    raw = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        text=True, stderr=subprocess.STDOUT
    )
    gpu_names = [x.strip() for x in raw.splitlines() if x.strip()]
except Exception as e:
    raise RuntimeError(f"Could not detect Kaggle GPUs: {e}")

print("Detected GPUs:", gpu_names)
is_dual_t4 = len(gpu_names) >= 2 and all("T4" in n.upper() for n in gpu_names[:2])
is_p100 = len(gpu_names) >= 1 and "P100" in gpu_names[0].upper()

if is_dual_t4:
    PROFILE = "DUAL_T4"
    print("✓ Profile: DUAL T4 — full-quality profile")
elif is_p100:
    PROFILE = "P100"
    print("✓ Profile: SINGLE P100 — safe 16 GB fallback")
else:
    PROFILE = "GENERIC_16GB"
    print("⚠ Unknown Kaggle GPU; using conservative 16 GB fallback:", gpu_names)

VENV = Path("/kaggle/working/lily_comfy_env")
VENV_PY = VENV / "bin" / "python"
if not VENV_PY.exists():
    print("Creating isolated ComfyUI environment...")
    subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", str(VENV)], check=True)
print("✓ Isolated ComfyUI environment ready")

SOURCE_URL = "https://raw.githubusercontent.com/benruiz1024-ops/hi/789b857e85738efdaec591f04de11bf76befe20d/LILY_WAN22_DUAL_T4_STUDIO.ipynb"
with urllib.request.urlopen(SOURCE_URL, timeout=60) as r:
    original = json.loads(r.read().decode("utf-8"))
cells = [c for c in original["cells"] if c.get("cell_type") == "code"]
if not cells:
    raise RuntimeError("Could not locate the studio code cell.")
code = "".join(cells[0]["source"])

bad_core = '''# Core helpers/UI.
pip_install("gradio>=5.20,<6", "huggingface_hub>=0.29", "requests>=2.32", "gdown>=5.2", "scikit-video")'''
safe_core = '''# Core helpers/UI — kernel-safe.
print("✓ Using Kaggle's existing notebook packages; no blanket upgrades")'''
if bad_core not in code:
    raise RuntimeError("Could not locate the unsafe core installer.")
code = code.replace(bad_core, safe_core, 1)

old_helper = '''def pip_install(*pkgs):
    run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *pkgs])'''
new_helper = '''def pip_install(*pkgs):
    run([str(VENV_PY), "-m", "pip", "install", "--disable-pip-version-check", "--no-input", "--prefer-binary", *pkgs], check=False)'''
if old_helper in code:
    code = code.replace(old_helper, new_helper, 1)

old_req = 'run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=COMFY)'
new_req = '''print("Installing ComfyUI dependencies in isolated env...")
run([str(VENV_PY), "-m", "pip", "install", "--disable-pip-version-check", "--no-input", "--prefer-binary", "-r", "requirements.txt"], cwd=COMFY, check=True)
print("✓ ComfyUI dependencies ready")'''
if old_req not in code:
    raise RuntimeError("Could not locate ComfyUI requirements installer.")
code = code.replace(old_req, new_req, 1)

old_mag = 'run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=False)'
new_mag = 'run([str(VENV_PY), "-m", "pip", "install", "--disable-pip-version-check", "--no-input", "--prefer-binary", "-r", str(req)], check=False)'
if old_mag in code:
    code = code.replace(old_mag, new_mag, 1)

old_launch = '        sys.executable, "main.py",'
new_launch = '        str(VENV_PY), "main.py",'
if old_launch not in code:
    raise RuntimeError("Could not locate ComfyUI launch interpreter.")
code = code.replace(old_launch, new_launch, 1)

rife_start = "# -------------------------\n# 4) Optional RIFE on GPU 1"
rife_end = "# -------------------------\n# 5) Start ComfyUI on GPU 0"
a, b = code.find(rife_start), code.find(rife_end)
if a == -1 or b == -1 or b <= a:
    raise RuntimeError("Could not locate RIFE block.")
no_rife = '''# -------------------------
# 4) RIFE deferred for reliable startup
# -------------------------
RIFE_READY = False
print("✓ RIFE deferred — ffmpeg interpolation enabled")

'''
code = code[:a] + no_rife + code[b:]

if PROFILE != "DUAL_T4":
    old_presets = '''PRESETS = {
    # Frame counts are 4k+1, as Wan expects.
    # Lower fps makes each native sequence ~5 seconds before interpolation.
    "⚡ Turbo":  {"frames": 61,  "steps": 12, "source_fps": 12.0, "cfg": 5.0},
    "✨ Normal": {"frames": 81,  "steps": 16, "source_fps": 16.0, "cfg": 5.0},
    "👑 Max":    {"frames": 121, "steps": 20, "source_fps": 24.0, "cfg": 5.0},
}
SIZES = {
    "Landscape 16:9": (1280, 704),
    "Portrait 9:16": (704, 1280),
}'''
    safe_presets = '''PRESETS = {
    # Conservative 16 GB single-GPU profile.
    "⚡ Turbo":  {"frames": 49, "steps": 10, "source_fps": 10.0, "cfg": 5.0},
    "✨ Normal": {"frames": 61, "steps": 12, "source_fps": 12.0, "cfg": 5.0},
    "👑 Max":    {"frames": 81, "steps": 16, "source_fps": 16.0, "cfg": 5.0},
}
SIZES = {
    "Landscape 16:9": (832, 480),
    "Portrait 9:16": (480, 832),
}'''
    if old_presets not in code:
        raise RuntimeError("Could not locate preset block for 16 GB fallback.")
    code = code.replace(old_presets, safe_presets, 1)

print(f"✓ Hardware profile selected: {PROFILE}")
print("✓ Kernel-safe dependency isolation applied")
print("✓ Starting Wan 2.2 setup...\n")

exec(compile(code, "LILY_WAN22_ADAPTIVE_V4", "exec"), globals(), globals())
